# nb_gold
Executa uma consulta Gold da pasta `consultas` (lakehouse Gold → Files/consultas/<gold>.sql) e grava o
resultado. A regra de negócio fica toda no .sql; este notebook só cuida de ler, filtrar e gravar.

Cabeçalho do .sql (linhas `-- @chave valor`):
- `@destino`: tabela Gold gravada (lakehouse Gold);
- `@chave`: coluna da Gold que identifica o registro de origem (apagar e reinserir por ela);
- `@incremental`: `apelido.coluna` da fonte que comanda o incremental e cujo valor vai para `@chave`;
- `@fontes`: `apelido=Lakehouse.tabela; ...`, as tabelas que a consulta usa pelo apelido.

Incremental: a fonte de `@incremental` traz só as linhas que chegaram desde a última execução
(`_dt_carga`); a Gold apaga esses registros e grava o recálculo. Sem linha nova, termina sem gravar.
`modo = COMPLETO`, ou a primeira execução, recalcula e regrava a tabela inteira.

In [ ]:
gold = "gold_fato_coberturas"
lote = "manual"
modo = "INCREMENTAL"      # INCREMENTAL | COMPLETO

In [ ]:
import os, json, datetime as dt
import pyarrow as pa
import duckdb
from deltalake import DeltaTable, write_deltalake

# Não renomear os lakehouses: o banco do SQL endpoint guarda o nome da criação e a sincronização falha
LH_BRONZE, LH_SILVER, LH_GOLD = "Bronze", "Silver", "Gold"
FUSO_BRT = dt.timezone(dt.timedelta(hours=-3))

_LOCAL = os.environ.get("DW_LOCAL_ROOT")  # usado só em testes fora do Fabric
if _LOCAL:
    def _opts():
        return None
    def caminho(lakehouse, tabela):
        return f"{_LOCAL}/{lakehouse}/{tabela}"
    def sair(valor):
        globals()["_EXIT_VALUE"] = valor
else:
    import notebookutils
    _WS = notebookutils.runtime.context["currentWorkspaceName"]
    def _opts():
        return {"bearer_token": notebookutils.credentials.getToken("storage"),
                "use_fabric_endpoint": "true"}
    def caminho(lakehouse, tabela):
        return f"abfss://{_WS}@onelake.dfs.fabric.microsoft.com/{lakehouse}.Lakehouse/Tables/{tabela}"
    def sair(valor):
        notebookutils.notebook.exit(valor)

def existe(lakehouse, tabela):
    try:
        DeltaTable(caminho(lakehouse, tabela), storage_options=_opts())
        return True
    except Exception:
        return False

def ler(lakehouse, tabela, filtros=None, colunas=None):
    dt_ = DeltaTable(caminho(lakehouse, tabela), storage_options=_opts())
    kw = {k: v for k, v in (("filters", filtros), ("columns", colunas)) if v}
    return dt_.to_pyarrow_table(**kw)

def gravar(lakehouse, tabela, dados, modo="append"):
    if modo == "append":
        acrescentar_colunas(lakehouse, tabela, dados.schema)
    kw = {"schema_mode": "overwrite"} if modo == "overwrite" else {}
    write_deltalake(caminho(lakehouse, tabela), dados, mode=modo,
                    storage_options=_opts(), **kw)

def acrescentar_colunas(lakehouse, tabela, schema):
    """Coluna nova na origem (ex.: campo novo da API): a tabela é regravada com a coluna, vazia nas linhas
    antigas, antes do append ou do MERGE. Não depende da versão do deltalake suportar evolução de schema."""
    if not existe(lakehouse, tabela):
        return
    atual = tabela_delta(lakehouse, tabela)
    nomes = {f.name for f in atual.schema().fields}
    novas = [c for c in schema if c.name not in nomes]
    if not novas:
        return
    dados = atual.to_pyarrow_table()
    for c in novas:
        dados = dados.append_column(c, pa.nulls(dados.num_rows, type=c.type))
    write_deltalake(caminho(lakehouse, tabela), dados, mode="overwrite", schema_mode="overwrite",
                    storage_options=_opts())
    print(f"{tabela}: colunas novas {[c.name for c in novas]}")

def tabela_delta(lakehouse, tabela):
    return DeltaTable(caminho(lakehouse, tabela), storage_options=_opts())

def para_arrow(rel):
    """Converte resultado DuckDB em pyarrow.Table (compatível com várias versões)."""
    r = rel.fetch_arrow_table() if hasattr(rel, "fetch_arrow_table") else rel.arrow()
    return r.read_all() if hasattr(r, "read_all") else r

def nome_bronze(tabela_destino):
    """ctl_tabelas guarda o nome base (fato_contas_receber_gs); cada camada recebe o seu prefixo."""
    return f"bronze_{tabela_destino}"

def nome_silver(tabela_destino):
    return f"silver_{tabela_destino}"

def agora_utc():
    return dt.datetime.now(dt.timezone.utc)

def watermark_gold(gold, fonte):
    """Último _dt_carga da Silver já levado para esta Gold (tabela ctl_gold na Bronze). As Golds são
    incrementais por esta marca, e não pelo lote: se uma Gold falhar, a próxima execução retoma dali."""
    if not existe(LH_BRONZE, "ctl_gold"):
        return None
    for r in ler(LH_BRONZE, "ctl_gold").to_pylist():
        if r["gold"] == gold and r["fonte"] == fonte:
            return r["ultimo_dt_carga"]
    return None

def gravar_watermark_gold(gold, fonte, valor):
    linha = pa.Table.from_pylist([{"gold": gold, "fonte": fonte, "ultimo_dt_carga": valor,
                                   "dt_atualizacao": agora_utc()}],
                                 schema=pa.schema([("gold", pa.string()), ("fonte", pa.string()),
                                                   ("ultimo_dt_carga", pa.timestamp("us", tz="UTC")),
                                                   ("dt_atualizacao", pa.timestamp("us", tz="UTC"))]))
    if not existe(LH_BRONZE, "ctl_gold"):
        gravar(LH_BRONZE, "ctl_gold", linha, modo="overwrite")
        return
    (tabela_delta(LH_BRONZE, "ctl_gold")
        .merge(linha, predicate="t.gold = s.gold AND t.fonte = s.fonte", source_alias="s", target_alias="t")
        .when_matched_update_all().when_not_matched_insert_all().execute())

def consulta(nome):
    """Texto de uma consulta da pasta `consultas` (no Fabric: lakehouse Gold → Files/consultas/<nome>.sql,
    copiada de implantacao/consultas pela implantação). Nos testes locais, lê de DW_CONSULTAS."""
    if _LOCAL:
        with open(os.path.join(os.environ["DW_CONSULTAS"], f"{nome}.sql"), encoding="utf-8") as arq:
            return arq.read()
    return notebookutils.fs.head(
        f"abfss://{_WS}@onelake.dfs.fabric.microsoft.com/{LH_GOLD}.Lakehouse/Files/consultas/{nome}.sql", 10_000_000)

def cabecalho(sql):
    """Configuração no topo do .sql, em linhas `-- @chave valor` (ex.: `-- @destino gold_fato_coberturas`)."""
    cfg = {}
    for linha in sql.splitlines():
        linha = linha.strip()
        if linha.startswith("-- @"):
            chave, _, valor = linha[4:].partition(" ")
            cfg[chave] = valor.strip()
    return cfg

def ler_config(nome):
    """Arquivo YAML da pasta config (no Fabric: lakehouse Gold → Files/config/<nome>, copiado de config/ pela
    implantação). Nos testes locais, lê de DW_CONFIG."""
    import yaml
    if _LOCAL:
        with open(os.path.join(os.environ["DW_CONFIG"], nome), encoding="utf-8") as arq:
            return yaml.safe_load(arq)
    return yaml.safe_load(notebookutils.fs.head(
        f"abfss://{_WS}@onelake.dfs.fabric.microsoft.com/{LH_GOLD}.Lakehouse/Files/config/{nome}", 10_000_000))

In [ ]:
import pyarrow.compute as pc

LAKEHOUSES = {"Bronze": LH_BRONZE, "Silver": LH_SILVER, "Gold": LH_GOLD}
inicio = agora_utc()
sql = consulta(gold)
cfg = cabecalho(sql)
for campo in ("destino", "chave", "incremental", "fontes"):
    if not cfg.get(campo):
        raise ValueError(f"{gold}.sql sem `-- @{campo}` no cabeçalho")
destino, chave = cfg["destino"], cfg["chave"]
apelido_inc, coluna_inc = cfg["incremental"].split(".", 1)
fontes = {}
for item in cfg["fontes"].split(";"):
    apelido, _, ref = item.strip().partition("=")
    lh, _, tabela = ref.strip().partition(".")
    fontes[apelido.strip()] = (LAKEHOUSES[lh], tabela)

lh_inc, tabela_inc = fontes[apelido_inc]
wm = watermark_gold(destino, tabela_inc)
completo = modo == "COMPLETO" or wm is None or not existe(LH_GOLD, destino)

con = duckdb.connect()
# as datas da Bronze são horário de Brasília rotulado UTC: converter em UTC para não recuar um dia
con.execute("SET TimeZone = 'UTC'").fetchall()
for apelido, (lh, tabela) in fontes.items():
    if apelido == apelido_inc and not completo:
        dados = ler(lh, tabela, filtros=[("_dt_carga", ">", wm)])
    else:
        dados = ler(lh, tabela)
    if apelido == apelido_inc:
        alterados = dados
    con.register(apelido, dados)

In [ ]:
if alterados.num_rows == 0:
    print(f"{destino}: nada novo")
    resultado = {"gold": destino, "registros": 0, "linhas": 0, "completo": False}
else:
    linhas = para_arrow(con.sql(sql))
    chaves = sorted(set(pc.unique(alterados.column(coluna_inc)).drop_null().to_pylist()))
    novo_wm = pc.max(alterados.column("_dt_carga")).as_py()
    con.close()
    if completo:
        if linhas.num_rows == 0:
            raise RuntimeError(f"{destino}: consulta sem linhas; tabela não substituída para não esvaziá-la")
        gravar(LH_GOLD, destino, linhas, modo="overwrite")
    else:
        alvo = tabela_delta(LH_GOLD, destino)
        for i in range(0, len(chaves), 5000):
            alvo.delete(f"{chave} IN ({', '.join(str(int(x)) for x in chaves[i:i + 5000])})")
        if linhas.num_rows:
            gravar(LH_GOLD, destino, linhas)
    gravar_watermark_gold(destino, tabela_inc, novo_wm)
    gravar_watermark_gold(destino, "escrita", inicio)
    duracao = (agora_utc() - inicio).total_seconds()
    print(f"{destino}: {'COMPLETO' if completo else 'INCREMENTAL'} | {len(chaves)} registros | "
          f"{linhas.num_rows} linhas | {duracao:.0f}s")
    resultado = {"gold": destino, "registros": len(chaves), "linhas": linhas.num_rows, "completo": completo}

In [ ]:
sair(json.dumps(resultado))